# 🧬 iMODFIT — Flexible Fitting of Atomic Structures into Cryo‑EM Maps

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ts387/imodfit/blob/claude/lucid-bell-M9Uxw/iMODFIT_Colab.ipynb)

**iMODFIT** (*internal coordinates normal MODe based FITting*) bends a starting
atomic model (a `.pdb`) so that it matches a target **cryo‑electron‑microscopy
density map**. It does this by deforming the structure along its low‑frequency
**normal modes computed in internal (dihedral) coordinates** — which keeps the
geometry physically reasonable while capturing large, collective motions
(domain closures, hinge bending, etc.).

This notebook gets you from *nothing* to *a fitted structure* in a few minutes,
entirely in the browser.

### What you'll do here
1. **Get iMODFIT** — clone this repo (pre‑compiled Linux binaries + test data).
2. **Install** the handful of system libraries it needs.
3. **Verify** the install.
4. **Run the bundled example** — fit an *open* GroEL subunit into a *closed* 10 Å map.
5. **Score it** — measure the C‑α RMSD against the known answer.
6. **Plot** the convergence (correlation vs. iteration).
7. **Visualize** initial → fitted → reference structures in 3D.
8. **Fit your own** structure — into a single map, *or batch‑fit it into many maps at once*.
9. *Appendices:* parameter reference, companion tools, and an optional faster build.

> **Runtime:** iMODFIT is CPU‑only — you do **not** need a GPU. The default
> Colab CPU runtime is fine. (`Runtime → Change runtime type → CPU`.)

> **Citation:** López‑Blanco JR & Chacón P. *iMODFIT: Efficient and robust
> flexible fitting based on vibrational analysis in internal coordinates.*
> J. Struct. Biol. (2013) 184(2):261‑270.

## 1 · Get iMODFIT

Everything we need — the executables (`bin/`) and the tutorial data
(`imodfit_test/`) — lives in this repository, so the first step is simply to
clone it. We also add `bin/` to the `PATH` so we can call the tools by name.

In [ ]:
import os

# --- Configuration (edit if you forked the repo or after merging to main) ---
REPO_URL = "https://github.com/ts387/imodfit.git"
BRANCH   = "claude/lucid-bell-M9Uxw"   # ← change to "main" once this is merged
REPO_DIR = "/content/imodfit"
# ---------------------------------------------------------------------------

if not os.path.isdir(REPO_DIR):
    print(f"Cloning {REPO_URL}  (branch: {BRANCH})")
    !git clone --depth 1 --branch {BRANCH} {REPO_URL} {REPO_DIR}
else:
    print("Repo already present — pulling latest.")
    !git -C {REPO_DIR} pull --ff-only

# Ensure the binaries are executable and reachable by name.
!chmod +x {REPO_DIR}/bin/*
os.environ["PATH"] = f"{REPO_DIR}/bin:" + os.environ["PATH"]

print("\nAvailable executables:")
!ls -lh {REPO_DIR}/bin

# 💡 If this repo is PRIVATE, the clone above will fail. In that case use a
#    GitHub token, e.g.:
#    REPO_URL = "https://<YOUR_TOKEN>@github.com/ts387/imodfit.git"

## 2 · Install the runtime libraries

We use the **GNU (`_gcc`) build**, which depends only on standard open‑source
math libraries — all available through `apt`:

| Library | Package | Used for |
|---------|---------|----------|
| LAPACK  | `liblapack3` | linear algebra |
| BLAS    | `libblas3`   | linear algebra |
| ARPACK  | `libarpack2` / `libarpack2t64` | eigen‑solver (normal modes) |
| FFTW (single precision) | `libfftw3-single3` | map filtering / correlation |

*(The package `libarpack2` was renamed to `libarpack2t64` in Ubuntu 24.04, so we
try both names.)*

In [ ]:
%%bash
echo "Installing LAPACK, BLAS, ARPACK and single-precision FFTW ..."
apt-get -qq update
apt-get -qq install -y liblapack3 libblas3 libfftw3-single3
# ARPACK: package name differs across Ubuntu releases — try both.
apt-get -qq install -y libarpack2t64 || apt-get -qq install -y libarpack2
echo "✅ Done."

## 3 · Verify the installation

If the iMODFIT welcome banner prints, the binary and all its libraries are
correctly wired up.

In [ ]:
!imodfit_gcc --help | head -n 5
print("\n✅ iMODFIT is ready to use.")

### How iMODFIT works & what it needs

**The loop:** at each step iMODFIT (1) computes the low‑frequency normal modes of
the *current* model in dihedral‑angle space, (2) moves the model a small step
along the combination of modes that best **increases the cross‑correlation** with
the target map, and (3) re‑diagonalizes periodically as the shape changes — until
the correlation converges.

**Command line:**

```
imodfit_gcc  <pdb>  <map>  <resolution>  <cutoff>  [options]
```

| Argument | Meaning |
|----------|---------|
| `<pdb>` | **Initial** atomic model to be deformed. |
| `<map>` | **Target** cryo‑EM map: `.ccp4`, `.mrc`, or Situs `.sit`. |
| `<resolution>` | Map resolution in **Å** (must match your map). |
| `<cutoff>` | Density **threshold**. `0` = use the whole map; otherwise set it to the map's recommended contour level to ignore solvent/noise. |

Handy options (full list via `imodfit_gcc --help`):

| Option | Effect |
|--------|--------|
| `-t` | Also write the trajectory `*_movie.pdb` (a "morph" you can play in VMD/ChimeraX). |
| `-o NAME` | Output basename (default `imodfit` → `imodfit_fitted.pdb`, …). |
| `-m {0,2}` | Model: `0` = Cα only (use if side chains are missing), `2` = heavy atoms (default). |
| `-n`, `-e` | How many normal modes / "excited" modes to use. |
| `-i N` | Maximum iterations. |
| `-r 0.7` | Randomly fix 70 % of dihedrals (faster, stiffer). |

## 4 · Run the bundled example — GroEL closing

The classic iMODFIT test case: take one subunit of GroEL in its **open**
conformation (`1sx4A.pdb`) and flexibly fit it into a **10 Å simulated map of
the closed** conformation (`1oel.ccp4`). The known answer is the closed
structure `1oel.pdb`, so afterwards we can measure exactly how well we did.

The command is literally the one from the official tutorial. ⏱️ It runs a few
thousand iterations and takes roughly **3–6 minutes** on a Colab CPU — watch the
`score`/`corr.` columns improve as it goes.

In [ ]:
%cd /content/imodfit/imodfit_test

#            <pdb>      <map>      <res> <cutoff>  options
!imodfit_gcc 1sx4A.pdb  1oel.ccp4   10     0       -t -o imodfit

print("\nGenerated files:")
!ls -lh imodfit_fitted.pdb imodfit_movie.pdb imodfit_score.txt

## 5 · How good is the fit? (C‑α RMSD)

Because we know the true target structure (`1oel.pdb`), we can compute the
**C‑α RMSD** between our fitted model and the answer. The `rmsd` tool reports
the RMSD both as‑is and after optimal superposition.

> Expected: about **1.4 Å** C‑α RMSD — a near‑perfect recovery of the open→closed
> transition. (The exact value varies slightly run‑to‑run because iMODFIT uses a
> random seed.)

In [ ]:
!rmsd_gcc imodfit_fitted.pdb 1oel.pdb -c

## 6 · Watch it converge

iMODFIT logs its progress to `imodfit_score.txt` (one row every few iterations).
The columns are `iteration  score  cross‑correlation  …`. A good fit shows the
**cross‑correlation rising toward 1.0** while the score drops.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Robust parser: most rows are "iter score corr dScore dAvg", but the first row
# has only 3 columns and the final (converged) row carries one extra field that
# shifts the correlation to column 3 — so we can't rely on np.loadtxt here.
it, score, cc = [], [], []
with open("imodfit_score.txt") as fh:
    for line in fh:
        if line.startswith("#") or not line.strip():
            continue
        f = line.split()
        corr = f[3] if len(f) >= 6 else f[2]   # extra column on the last row
        it.append(float(f[0])); score.append(float(f[1])); cc.append(float(corr))
it, score, cc = np.array(it), np.array(score), np.array(cc)

fig, ax1 = plt.subplots(figsize=(8, 4.5))
ax1.plot(it, cc, color="tab:blue", lw=2)
ax1.set_xlabel("iteration")
ax1.set_ylabel("map cross-correlation", color="tab:blue")
ax1.tick_params(axis="y", labelcolor="tab:blue")
ax1.grid(alpha=0.3)

ax2 = ax1.twinx()
ax2.plot(it, score, color="tab:red", lw=1.5, ls="--")
ax2.set_ylabel("score (lower = better)", color="tab:red")
ax2.tick_params(axis="y", labelcolor="tab:red")

plt.title("iMODFIT convergence")
fig.tight_layout()
plt.show()

print(f"Cross-correlation: {cc[0]:.4f}  →  {cc[-1]:.4f}   over {int(it[-1])} iterations")

## 7 · Visualize the fit in 3D

Let's overlay the three structures with `py3Dmol`. The **blue** fitted model
should swing away from the **salmon** starting model and land almost exactly on
the **green** reference — that's the conformational change iMODFIT discovered.

| Color | Structure |
|-------|-----------|
| 🔴 salmon  | `1sx4A.pdb` — initial (open) |
| 🔵 blue    | `imodfit_fitted.pdb` — iMODFIT result |
| 🟢 green   | `1oel.pdb` — reference (closed, the answer) |

In [ ]:
!pip -q install py3Dmol
import py3Dmol

def _read(path):
    with open(path) as fh:
        return fh.read()

view = py3Dmol.view(width=820, height=540)
view.addModel(_read("1sx4A.pdb"), "pdb")            # 0: initial
view.setStyle({"model": 0}, {"cartoon": {"color": "salmon"}})
view.addModel(_read("imodfit_fitted.pdb"), "pdb")   # 1: fitted
view.setStyle({"model": 1}, {"cartoon": {"color": "skyblue"}})
view.addModel(_read("1oel.pdb"), "pdb")             # 2: reference
view.setStyle({"model": 2}, {"cartoon": {"color": "palegreen"}})
view.setBackgroundColor("white")
view.zoomTo()
view.show()

## 8 · Fit your own structure and map

To fit your own data you need just two files:

1. **Initial model** — a `.pdb` (your starting conformation / homology model).
2. **Target map** — a cryo‑EM density map in `.ccp4`, `.mrc`, or Situs `.sit`.

…plus two numbers: the **resolution** (Å) and a **density cutoff**.

> **Picking the cutoff:** `0` tells iMODFIT to use the entire map (fine to start
> with). For noisy experimental maps, set it to the map's recommended contour
> level (the threshold at which the map looks "right" in ChimeraX) so background
> density is ignored.

> ⚠️ **Format:** iMODFIT reads **legacy PDB only** (`.pdb` / `.ent`) — *not*
> mmCIF (`.cif`). If your structure is mmCIF, convert it in **Step 8b** below.
> Also note: the default heavy‑atom model needs complete residues; for
> incomplete structures or homology models use the **Cα model** (`-m 0`), which
> only requires Cα atoms.

**Step 8a — upload your files** (a structure + a map; the structure may be a
`.cif` — we'll convert it next):

In [ ]:
import os
from google.colab import files

os.makedirs("/content/myrun", exist_ok=True)
%cd /content/myrun

print("Select your initial structure and your target map to upload…")
uploaded = files.upload()
print("\nUploaded:", list(uploaded.keys()))

**Step 8b — (optional) convert mmCIF → PDB.** *Skip this if you already uploaded
a `.pdb`.* iMODFIT can't read `.cif`, so if your structure is mmCIF, point
`CIF_IN` at it below — we use [`gemmi`](https://gemmi.readthedocs.io) to write a
clean, column‑correct PDB, then use `PDB_OUT` as your model in the next step.

In [ ]:
!pip -q install gemmi
import gemmi

# ===== EDIT THESE =====
CIF_IN  = "my_model.cif"     # your uploaded mmCIF / PDBx file
PDB_OUT = "my_model.pdb"     # PDB to write (use this as PDB in the next cell)
# ======================

st = gemmi.read_structure(CIF_IN)
st.setup_entities()          # tidy chains/entities for a well-formed PDB
st.write_pdb(PDB_OUT)

n_chains = len(st[0])
n_atoms  = sum(len(res) for chain in st[0] for res in chain)
print(f"✅ Wrote {PDB_OUT}: {n_atoms} atoms across {n_chains} chain(s).")
if n_atoms > 99999 or any(len(ch.name) > 1 for ch in st[0]):
    print("⚠️  This structure exceeds PDB limits (99,999 atoms / 1-char chain IDs).\n"
          "   Consider fitting it chain-by-chain or splitting it into sub-assemblies.")

**Step 8c — set the parameters and run.** Edit the four values below to match
your filenames and your map, then run the cell. (If you converted a `.cif` above,
set `PDB` to your `PDB_OUT`.)

In [ ]:
# ===================== EDIT THESE =====================
PDB        = "my_model.pdb"     # your starting structure
MAP        = "my_map.ccp4"      # your target cryo-EM map (.ccp4/.mrc/.sit)
RESOLUTION = 10                 # map resolution in Angstrom
CUTOFF     = 0                  # density threshold (0 = whole map; else contour level)
EXTRA      = "-t -o myfit"      # -t = save movie ; -o = output basename
# ======================================================

!imodfit_gcc {PDB} {MAP} {RESOLUTION} {CUTOFF} {EXTRA}

**Step 8d — download your results** (the fitted model, and optionally the movie):

In [ ]:
from google.colab import files
files.download("myfit_fitted.pdb")
# files.download("myfit_movie.pdb")   # uncomment to also grab the trajectory

## 9 · Batch mode — fit one model into many maps

Got several maps (e.g. different conformational states, time points, or
resolutions of the same complex)? Upload **one model** and **as many maps as you
like**, and iMODFIT will fit the model into each of them in turn.

**Outputs are named after each map**, so results never collide and are easy to
match up:

| Uploaded map | Fitted model | Trajectory | Score log |
|--------------|--------------|------------|-----------|
| `stateA.ccp4` | `stateA_fitted.pdb` | `stateA_movie.pdb` | `stateA_score.txt` |
| `stateB.mrc`  | `stateB_fitted.pdb` | `stateB_movie.pdb` | `stateB_score.txt` |

> ⏱️ Each map is a full fit (a few minutes), so the total time scales with the
> number of maps. The loop keeps going even if one map fails, and prints a
> summary at the end.

> The model must be a **`.pdb`** (not `.cif`) — convert it first with the `gemmi`
> snippet in **Step 8b** if needed.

**Step 9a — upload one model and one-or-more maps:**

In [ ]:
import os
from google.colab import files

os.makedirs("/content/batch", exist_ok=True)
%cd /content/batch

print("Upload ONE model (.pdb) and ONE OR MORE target maps (.ccp4/.mrc/.map/.sit)...")
uploaded = files.upload()
print("\nUploaded:", list(uploaded.keys()))

**Step 9b — set shared parameters and run the batch.** All maps are fitted with
the same `RESOLUTION`/`CUTOFF`; if your maps differ, just run this section once
per group. The model is auto‑detected from the uploaded `.pdb` (or set `MODEL`
explicitly).

In [ ]:
import os, shlex, subprocess

# ===================== EDIT THESE =====================
MODEL      = ""        # starting .pdb; leave "" to auto-detect the uploaded one
RESOLUTION = 10        # map resolution in Angstrom (shared by all maps)
CUTOFF     = 0         # density threshold (0 = whole map; else contour level)
EXTRA      = "-t"      # extra imodfit options ('-t' saves a movie). Do NOT add -o.
MAP_EXTS   = (".ccp4", ".mrc", ".map", ".sit", ".situs", ".vol")
# ======================================================

files_here = sorted(os.listdir("."))
if not MODEL:
    pdbs = [f for f in files_here if f.lower().endswith(".pdb")]
    assert pdbs, "No .pdb found - upload a model or set MODEL explicitly."
    MODEL = pdbs[0]
maps = [f for f in files_here if f.lower().endswith(MAP_EXTS)]
assert maps, "No maps found - upload at least one .ccp4/.mrc/.map/.sit file."

print(f"Model : {MODEL}")
print(f"Maps  : {maps}")
print(f"Naming: <map rootname>_fitted.pdb  (e.g. {maps[0]} -> "
      f"{os.path.splitext(maps[0])[0]}_fitted.pdb)\n")

def final_cc(score_file):
    \"\"\"Last cross-correlation from a score file (handles the ragged columns).\"\"\"
    last = None
    try:
        with open(score_file) as fh:
            for line in fh:
                if line.startswith("#") or not line.strip():
                    continue
                f = line.split()
                last = float(f[3] if len(f) >= 6 else f[2])
    except FileNotFoundError:
        pass
    return last

results = []
for i, m in enumerate(maps, 1):
    root = os.path.splitext(m)[0]                 # stateA.ccp4 -> stateA
    cmd = (["imodfit_gcc", MODEL, m, str(RESOLUTION), str(CUTOFF), "-o", root]
           + shlex.split(EXTRA))
    print(f"[{i}/{len(maps)}] fitting {MODEL} into {m}  ->  {root}_fitted.pdb ...")
    with open(f"{root}_imodfit.log", "w") as logf:
        rc = subprocess.run(cmd, stdout=logf, stderr=subprocess.STDOUT).returncode
    ok = (rc == 0) and os.path.exists(f"{root}_fitted.pdb")
    cc = final_cc(f"{root}_score.txt")
    status = "ok" if ok else f"FAILED(rc={rc})"
    ccs = f"{cc:.4f}" if cc is not None else "n/a"
    print(f"      {status}  (cross-corr = {ccs})")
    results.append((m, f"{root}_fitted.pdb", cc, status))

print("\n==================== Batch summary ====================")
print(f"{'map':<22}{'fitted model':<26}{'corr':>6}  status")
print("-" * 62)
for m, out, cc, st in results:
    print(f"{m:<22}{out:<26}{(f'{cc:.4f}' if cc is not None else '  -- '):>6}  {st}")

**Step 9c — download everything.** Bundle all fitted models (plus movies and
score logs) into a single zip.

In [ ]:
import glob, zipfile
from google.colab import files

patterns = ["*_fitted.pdb", "*_movie.pdb", "*_score.txt", "*_imodfit.log"]
to_zip = sorted({f for p in patterns for f in glob.glob(p)})
print(f"Bundling {len(to_zip)} files into batch_results.zip ...")
with zipfile.ZipFile("batch_results.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in to_zip:
        z.write(f)

files.download("batch_results.zip")
# Or grab a single fitted model, e.g.:
# files.download("stateA_fitted.pdb")

## Appendix A · Parameters & practical tips

- **Resolution & cutoff are the two knobs that matter most.** Always pass the
  real map resolution; start with `cutoff = 0`, then tighten it to the map's
  contour level if the fit is distracted by background density.
- **Missing side chains?** Use `-m 0` (Cα model). Full‑atom input → default
  `-m 2` is fine; add `-F` for full‑atom output models.
- **Big / symmetric complexes:** fit a single subunit into the corresponding
  region, or increase the modes used with `-n`. Use `-r 0.5`–`0.8` to randomly
  fix a fraction of dihedrals for speed/stability.
- **Multi‑chain inputs** are supported; inter‑chain rigid‑body motions are
  handled with 6 extra coordinates per chain.
- **Reproducibility:** pass `--seed <int>` to make a run deterministic.
- **Outputs:** `*_fitted.pdb` (final model), `*_movie.pdb` (trajectory, with
  `-t`), `*_score.txt` (convergence log).
- See `imodfit_gcc --help` for the complete option list.

## Appendix B · The companion tools

This release ships three helper programs (all on your `PATH` already):

- **`pdb2vol_gcc`** — simulate a density map from a PDB. Great for making test
  targets or sanity‑checking a fit.
  `pdb2vol_gcc <pdb> <out_map> <resolution> <voxel_size>`
- **`rmsd_gcc`** — optimal superposition & RMSD between two structures
  (`-c` = C‑α only, `-b` = backbone).
- **`pdbtool_gcc`** — atomic‑structure manipulation utility.

The cell below uses `pdb2vol` to simulate a 10 Å map from the GroEL reference
structure — exactly the kind of map iMODFIT fits against.

In [ ]:
%cd /content/imodfit/imodfit_test
# pdb2vol_gcc  <pdb>     <out_map>          <resolution> <voxel_size>
!pdb2vol_gcc   1oel.pdb  simulated_10A.ccp4   10           2.0
!ls -lh simulated_10A.ccp4

## Appendix C · (Optional, advanced) Use the faster Intel‑MKL build

`imodfit_mkl` is the Intel‑compiler build and is noticeably faster, but it needs
the **Intel Math Kernel Library** at runtime. The GCC build you've used so far is
perfectly fine — only bother with this if you're running many/large fits.

The cell installs MKL via `pip`, creates the unversioned library names the binary
expects, and points the loader at them. If anything here fails, just keep using
`imodfit_gcc`.

In [ ]:
import os, glob, subprocess

!pip -q install mkl

# pip's MKL libs land in different places across environments, so locate them.
found = subprocess.run(
    ["bash", "-lc", "find /usr -name 'libmkl_core.so.*' 2>/dev/null | head -1"],
    capture_output=True, text=True).stdout.strip()

if not found:
    print("Couldn't find the MKL libraries - just keep using imodfit_gcc.")
else:
    libdir = os.path.dirname(found)
    print("MKL libraries found in:", libdir)
    # The binary asks for the unversioned names (...lp64.so) -> make those symlinks.
    for stem in ("libmkl_intel_lp64", "libmkl_sequential", "libmkl_core"):
        hits = sorted(glob.glob(os.path.join(libdir, stem + ".so.*")))
        link = os.path.join(libdir, stem + ".so")
        if hits and not os.path.exists(link):
            os.symlink(hits[0], link)
            print("  linked", os.path.basename(link), "->", os.path.basename(hits[0]))
    os.environ["LD_LIBRARY_PATH"] = libdir + ":" + os.environ.get("LD_LIBRARY_PATH", "")

    print("\nMKL build self-test:")
    !imodfit_mkl --help | head -n 3
    print("\nIf you saw the banner, swap 'imodfit_gcc' -> 'imodfit_mkl' above for more speed.")

---

### Credits & links
- **iMODFIT** — Structural Bioinformatics Group, IQFR‑CSIC (Madrid):
  José Ramón López‑Blanco & Pablo Chacón.
- Tutorials & more tools: <http://chaconlab.org/methods/fitting/imodfit>
- **Please cite:** López‑Blanco JR & Chacón P., *J. Struct. Biol.* (2013)
  184(2):261‑270. doi:[10.1016/j.jsb.2013.08.010](https://doi.org/10.1016/j.jsb.2013.08.010)